# Topologická optimalizace výztuhy

## Zadání slovy

> Odlehčená výztuha — držák implantátu nebo rám vnějšího fixátoru. Vlevo je
> pevně přichycená ke kosti, vpravo na ni působí síla. Obdélníkový prostor, do
> kterého se smí stavět, je daný anatomií. Materiálu je ale rozpočet: smí se
> použít nejvýše 35 % objemu toho obdélníku, protože zbytek je hmotnost navíc.
>
> **Kam ten materiál dát, aby byla výztuha co nejtužší?**

Přesně takhle dnes vznikají 3D tištěné titanové implantáty a nosné díly
v letectví: konstruktér zadá zástavbový prostor, uchycení, zatížení a rozpočet
hmotnosti — a tvar dopočítá optimalizace. Výsledek pak často vypadá jako kost,
protože kost řeší tutéž úlohu.

## Formulace

Deska se rozdělí na $N$ čtvercových elementů. Proměnná $x_e$ je **hustota
materiálu** v elementu $e$: $0$ je díra, $1$ plný titan.

$$
\begin{aligned}
\text{minimize}_{\mathbf x}\quad & c(\mathbf x)=\mathbf f^{\top}\mathbf u
  =\sum_{e=1}^{N} E_e(x_e)\,\mathbf u_e^{\top}K_0\,\mathbf u_e
  && \text{compliance}\\
\text{subject to}\quad & K(\mathbf x)\,\mathbf u=\mathbf f
  && \text{rovnovaha (MKP)}\\
& \frac{1}{N}\sum_{e=1}^{N} x_e \le \texttt{VOLFRAC}
  && \text{rozpocet materialu}\\
& 0\le x_e\le 1,\qquad
  E_e(x_e)=E_{\min}+x_e^{\,p}\,(E_0-E_{\min})
  && \text{SIMP}
\end{aligned}
$$

Účelová funkce $c$ je **poddajnost** (*compliance*), tedy práce, kterou síla
vykoná na posunutí; malá poddajnost znamená velkou tuhost. Omezení
$K(\mathbf x)\mathbf u=\mathbf f$ je fyzika — soustava rovnic z metody konečných
prvků, která se musí vyřešit v každé iteraci znovu, protože matice tuhosti $K$
závisí na proměnných.

Exponent $p = 3$ je trik metody **SIMP**: poloviční hustota dá jen osminovou
tuhost, takže se šedé mezihodnoty nevyplatí a řešení samo sklouzne k „buď
materiál, nebo díra“. Tím se ale úloha stane **nekonvexní** a konvexní solver ji
nevezme; hustoty se místo toho aktualizují metodou *optimality criteria*, což je
jednoduchá formule odvozená z KKT podmínek. Ke spádu se navíc přidává **filtr**
přes okolí o poloměru `RMIN` elementů — bez něj vyjde šachovnice, která je
v diskretizaci uměle tuhá.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| hustoty $x_e$ | pole `x` tvaru (NELY, NELX) |
| $E_e(x_e)$, SIMP | `E = EMIN + x.ravel()**PENAL * (E0 - EMIN)` |
| $K(\mathbf x)\mathbf u = \mathbf f$ | `u[volne] = spsolve(K[volne][:, volne], f[volne])` |
| $c(\mathbf x)=\mathbf f^\top\mathbf u$ | návratová hodnota `poddajnost(x)` |
| $\partial c/\partial x_e$ | `dc = -PENAL * x**(PENAL - 1) * (E0 - EMIN) * ce` |
| rozpočet materiálu | půlení násobitele v `krok_oc()` |

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_topologie.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_topologie.py) v repozitáři předmětu.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import convolve2d
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve

## Síť konečných prvků

Tahle buňka je standardní příprava MKP (matice tuhosti čtvercového elementu,
očíslování posunů, okrajové podmínky, jádro filtru) — je to řemeslo, které se
v každé implementaci opisuje stejně. Optimalizace sama začíná až v další buňce.

In [ ]:
VOLFRAC = 0.35  # @param {type:"slider", min:0.1, max:0.7, step:0.05}
ITERACI = 30  # @param {type:"slider", min:5, max:60, step:5}
RMIN = 2.0  # @param {type:"slider", min:1, max:4, step:0.5}
PENAL = 3.0  # @param {type:"slider", min:1, max:5, step:0.5}
NELX, NELY, E0, EMIN, NU, KROK = 60, 30, 1.0, 1e-9, 0.3, 0.2
POCET, ndof = NELX * NELY, 2 * (NELX + 1) * (NELY + 1)

k = np.array([1/2 - NU/6, 1/8 + NU/8, -1/4 - NU/12, -1/8 + 3*NU/8,
              -1/4 + NU/12, -1/8 - NU/8, NU/6, 1/8 - 3*NU/8])
KE = k[np.array([[0, 1, 2, 3, 4, 5, 6, 7], [1, 0, 7, 6, 5, 4, 3, 2], [2, 7, 0, 5, 6, 3, 4, 1],
                 [3, 6, 5, 0, 7, 2, 1, 4], [4, 5, 6, 7, 0, 1, 2, 3], [5, 4, 3, 2, 1, 0, 7, 6],
                 [6, 3, 4, 1, 2, 7, 0, 5], [7, 2, 1, 4, 3, 6, 5, 0]])] / (1 - NU**2)

e = np.arange(POCET)
rohy = ((e // NELX) * (NELX + 1) + e % NELX)[:, None] + [0, 1, NELX + 2, NELX + 1]
edof = np.repeat(2 * rohy, 2, 1); edof[:, 1::2] += 1      # dva posuny na uzel
iK, jK = np.repeat(edof, 8, 1).ravel(), np.tile(edof, 8).ravel()

f = np.zeros(ndof); f[2 * ((NELY // 2) * (NELX + 1) + NELX) + 1] = -1.0   # síla vpravo dolů
volne = np.setdiff1d(np.arange(ndof), 2*np.arange(NELY+1)[:, None]*(NELX+1) + [0, 1])
d = np.arange(-int(RMIN), int(RMIN) + 1)
jadro = np.maximum(0.0, RMIN - np.hypot(*np.meshgrid(d, d)))   # kuželové váhy filtru
Hs = convolve2d(np.ones((NELY, NELX)), jadro, mode="same")
print(f"{POCET} elementů, {ndof} stupňů volnosti, rozpočet {100 * VOLFRAC:.0f} % materiálu")

## Model a řešení

Jedna iterace má tři kroky: vyřešit rovnováhu při současném rozložení materiálu,
spočítat citlivost poddajnosti na každou hustotu a materiál podle ní přerozdělit
tak, aby se rozpočet vyčerpal přesně.

In [ ]:
def poddajnost(x):
    E = EMIN + x.ravel()**PENAL * (E0 - EMIN)             # SIMP: šedé hustoty se penalizují
    K = coo_matrix(((KE.ravel()[:, None] * E).ravel(order="F"), (iK, jK)),
                   shape=(ndof, ndof)).tocsc()
    u = np.zeros(ndof)
    u[volne] = spsolve(K[volne][:, volne], f[volne])
    ce = (u[edof] @ KE * u[edof]).sum(1)                  # energie napjatosti elementu
    return (E * ce).sum(), ce.reshape(NELY, NELX)


def krok_oc(x, dc):
    l1, l2 = 0.0, 1e9
    while (l2 - l1) / (l1 + l2 + 1e-12) > 1e-4:           # půlení: stínová cena materiálu
        lm = 0.5 * (l1 + l2)
        xn = np.clip(x * np.sqrt(-dc / lm), np.maximum(0.0, x - KROK), np.minimum(1.0, x + KROK))
        l1, l2 = (lm, l2) if xn.mean() > VOLFRAC else (l1, lm)
    return xn


x = np.full((NELY, NELX), VOLFRAC)                        # start: materiál rovnoměrně
c_rovnomerna = poddajnost(x)[0]
for it in range(ITERACI):
    c, ce = poddajnost(x)                                             # 1) rovnováha K(x) u = f
    dc = -PENAL * x**(PENAL - 1) * (E0 - EMIN) * ce                   # 2) citlivost poddajnosti
    dc = convolve2d(x * dc, jadro, mode="same") / Hs / np.maximum(1e-3, x)   # filtr citlivosti
    x = krok_oc(x, dc)                                                # 3) přerozdělení materiálu
c_opt, _ = poddajnost(x)

print(f"rovnoměrná deska: poddajnost {c_rovnomerna:.1f}")
print(f"optimalizovaný tvar téže hmotnosti: {c_opt:.1f} → {c_rovnomerna / c_opt:.1f}× tužší")

## Kontrola, která umí selhat

Optimality criteria není solver s certifikátem, takže se ověřuje ručně: rozpočet
materiálu musí být vyčerpaný přesně (jinak by šlo přidat materiál a být tužší)
a optimalizovaný tvar musí být tužší než rovnoměrná deska téže hmotnosti.

In [ ]:
assert abs(x.mean() - VOLFRAC) < 1e-3, f"rozpočet nesedí: {x.mean():.4f}"
assert c_opt < c_rovnomerna, "optimalizace nezlepšila ani rovnoměrnou desku"
print(f"průměrná hustota {x.mean():.4f}, rozpočet {VOLFRAC:.2f} — omezení je aktivní")

## Obrázek

In [ ]:
plt.imshow(x, cmap="gray_r", vmin=0, vmax=1)
plt.title(f"{100 * VOLFRAC:.0f} % materiálu, poddajnost {c_opt:.1f}")
plt.axis("off")
plt.show()

## Na co se zeptat kódu

1. `VOLFRAC = 0.15` a `0.60` — změní se jen tloušťka žeber, nebo i jejich počet?
2. `RMIN = 1.0` — objeví se šachovnice s *lepší* poddajností. Proč je to artefakt
   sítě?
3. `PENAL = 1.0` — bez penalizace je úloha v hustotách konvexní. Jde výsledek
   vyrobit?